# LangGraph Customer Support Agent with Snowflake AI Observability

This notebook demonstrates how to:

1. **Build a LangGraph agent** that uses Snowflake Cortex Search and Cortex Analyst as tools
2. **Instrument the agent with TruLens** (`TruGraph`) for full trace capture
3. **Run evaluations** using Snowflake AI Observability to measure answer relevance, context relevance, and groundedness

### Architecture

```
User Query
    │
    ▼
LangGraph ReAct Agent (ChatSnowflake LLM)
    ├── Tool: Cortex Search (unstructured case lookups)
    └── Tool: Cortex Analyst (structured metrics via semantic view)
    │
    ▼
TruGraph (auto-instrumentation) → Snowflake AI Observability
```

### Prerequisites

- Run `setup.sql` to create the `CUST_SUPPORT_DEMO.SUPPORT` schema, tables, semantic view, and Cortex Search service
- Snowflake account with Cortex features enabled
- `CORTEX_USER` database role and `CREATE EXTERNAL AGENT` / `CREATE TASK` / `EXECUTE TASK` privileges

In [1]:
%pip install --quiet -U langchain-core langchain-snowflake langgraph trulens-core trulens-providers-cortex trulens-connectors-snowflake trulens-apps-langgraph

## 1. Snowflake Session Setup

In [2]:
import os

sorted(os.environ)[-30:]

In [ ]:
import os

        # "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        # "user": os.getenv("SNOWFLAKE_USER"),
        # "password": os.getenv("SNOWFLAKE_PAT"), 
        # "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH"),
        # "database": DATABASE,
        # "schema": SCHEMA,
        # "role": ROLE


os.environ["SNOWFLAKE_ACCOUNT"] = "<your_account>"
os.environ["SNOWFLAKE_USER"] = "<your_username>"
os.environ["SNOWFLAKE_PASSWORD"] = "<your_password>"
os.environ["SNOWFLAKE_WAREHOUSE"] = "COMPUTE_WH"
os.environ["SNOWFLAKE_DATABASE"] = "CUST_SUPPORT_DEMO"
os.environ["SNOWFLAKE_SCHEMA"] = "AGENTS"

from langchain_snowflake import create_session_from_env

session = create_session_from_env()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

## 2. Define Snowflake Cortex Tools

In [4]:
import json
import textwrap
from langchain_core.tools import tool
from langchain_snowflake import SnowflakeCortexSearchRetriever

from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

CORTEX_SEARCH_SERVICE = "CUST_SUPPORT_DEMO.AGENTS.CASE_SEARCH_SERVICE"

retriever = SnowflakeCortexSearchRetriever(
    session=session,
    schema="CUST_SUPPORT_DEMO.AGENTS",
    service_name=CORTEX_SEARCH_SERVICE,
    k=5,
    auto_format_for_rag=True,
    content_field="ISSUE_SUMMARY",
    join_separator="\n\n",
    fallback_to_page_content=True,
)


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def search_support_cases(query: str) -> list:
    """Search support case details, issue descriptions, and resolution summaries.
    Use for questions about specific issues, finding cases by topic or keyword,
    or looking up resolution steps for a type of problem."""
    docs = retriever.invoke(query)
    # return "\n\n".join(doc.page_content for doc in docs)
    return [doc.page_content for doc in docs]

print("Cortex Search tool defined.")
test_results = search_support_cases.invoke("authentication login failures")
# print(textwrap.shorten(test_results, width=200, placeholder="..."))

In [5]:
import requests

SEMANTIC_VIEW = "CUST_SUPPORT_DEMO.AGENTS.SUPPORT_ANALYTICS"


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "question",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def query_support_analytics(question: str) -> list:
    """Query structured support metrics using natural language.
    Use for quantitative questions about case counts, CSAT scores, resolution times,
    escalation rates, agent performance comparisons, and trend analysis."""
    host = session.connection.host
    token = session.connection.rest.token
    # token = os.getenv("SNOWFLAKE_PAT")

    resp = requests.post(
        url=f"https://{host}/api/v2/cortex/analyst/message",
        json={
            "messages": [
                {"role": "user", "content": [{"type": "text", "text": question}]}
            ],
            "semantic_view": SEMANTIC_VIEW,
        },
        headers={
            "Authorization": f'Snowflake Token="{token}"',
            "Content-Type": "application/json",
        },
    )
    resp.raise_for_status()
    data = resp.json()

    result_parts = []
    msg = data.get("message") or {}
    for block in msg.get("content", []):
        if block.get("type") == "text":
            result_parts.append(block["text"])
        elif block.get("type") == "sql":
            result_parts.append(f"SQL: {block['statement']}")
            sql_results = session.sql(block["statement"]).collect()
            result_parts.append(json.dumps([row.as_dict() for row in sql_results[:20]], default=str))
        elif block.get("type") == "result_table":
            result_parts.append(json.dumps(block.get("data", [])[:20], default=str))
    # return "\n".join(result_parts) if result_parts else "No results returned."
    return result_parts

print("Cortex Analyst tool defined.")
test_analytics = query_support_analytics.invoke("How many total support cases are there?")
print(test_analytics[:300])

## 3. Build the LangGraph Agent

In [9]:
from langchain_snowflake import ChatSnowflake
from langgraph.prebuilt import create_react_agent

llm = ChatSnowflake(
    session=session,
    # model="claude-4-sonnet",
    model="claude-sonnet-4-6",
    temperature=0,
    max_tokens=2048,
)

SYSTEM_PROMPT = """You are a customer support analytics assistant for a SaaS company.
You have access to two tools:

1. **query_support_analytics** - For quantitative questions about metrics, trends, counts,
   averages, or comparisons. This queries structured data via Cortex Analyst.

2. **search_support_cases** - For questions about specific case details, issue descriptions,
   resolution steps, or searching cases by topic. This searches unstructured case text via Cortex Search.

Be concise and data-driven. When presenting numbers, use appropriate formatting.
If a question involves both metrics and case details, use both tools."""

tools = [search_support_cases, query_support_analytics]

graph = create_react_agent(
    llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print(f"LangGraph agent compiled. Nodes: {list(graph.get_graph().nodes.keys())}")

## 4. Test the Agent

In [10]:
response = graph.invoke(
    {"messages": [("user", "What is the average CSAT score by product?")]}
)
print(response["messages"][-1].content)

In [11]:
response = graph.invoke(
    {"messages": [("user", "Find cases related to authentication failures and how they were resolved")]}
)
print(response["messages"][-1].content)

## 5. Instrument with TruLens + Snowflake AI Observability

We wrap the LangGraph agent with `TruGraph` which:
- Auto-instruments all graph nodes and tool calls
- Captures full execution traces (inputs, outputs, latency)
- Exports traces to Snowflake for the AI Observability UI

In [ ]:
# APP_NAME = "CUSTOMER_SUPPORT_AGENT_LANGGRAPH"

# session.sql(f'DROP EXTERNAL AGENT {APP_NAME}').collect()

In [12]:
from trulens.apps.langgraph import TruGraph
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

APP_NAME = "CUSTOMER_SUPPORT_AGENT_LANGGRAPH"
APP_VERSION = "LANGGRAPH_V1"

tru_graph = TruGraph(
    graph,
    app_name=APP_NAME,
    app_version=APP_VERSION,
    connector=tru_snowflake_connector,
    main_method = graph.invoke
)

print(f"TruGraph registered: {APP_NAME} / {APP_VERSION}")

## 6. Define Evaluation Dataset

We create a test dataset that exercises both tools across different query types.

In [ ]:
import pandas as pd

queries_df = session.table("CUST_SUPPORT_DEMO.AGENTS.EVAL_DATA").to_pandas()

queries_df['query_json'] = queries_df['INPUT_QUERY'].apply(lambda x: {"messages": [("user", x)]})
queries_df['ground_truth_string'] = queries_df['GROUND_TRUTH_DATA'].apply(lambda x: json.loads(x).get('ground_truth_output'))
print(f"Evaluation dataset: {len(queries_df)} queries")
queries_df

## 7. Execute Evaluation Run

In [14]:
from trulens.core.run import Run
from trulens.core.run import RunConfig
import datetime

run_version = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

run_config = RunConfig(
    run_name=f"batch_run_{run_version}",
    description="Questions for Support Agent",
    dataset_name="TEST_QUERIES_DF",
    source_type="DATAFRAME",
    label="BATCH",
    dataset_spec={
        "RECORD_ROOT.INPUT": "query_json",
        "RECORD_ROOT.GROUND_TRUTH_OUTPUT": "ground_truth_string",

    },
    
)

In [15]:
run = tru_graph.add_run(run_config=run_config)

run.start(input_df=queries_df)

## 8. Compute Evaluation Metrics

We compute three key RAG evaluation metrics:
- **Answer Relevance**: Is the response relevant to the user's query?
- **Context Relevance**: Is the retrieved context relevant to the query?
- **Groundedness**: Is the response grounded in the retrieved context?

In [17]:
import time

metric_list = [
    "correctness",
    "groundedness",
    "coherence",
]

run.compute_metrics(metric_list)
print("Metrics computation started...")

while run.get_status() not in ("COMPLETED", "PARTIALLY_COMPLETED", "CANCELLED"):
    print(f"  Status: {run.get_status()}")
    time.sleep(5)

print(f"Evaluation complete. Final status: {run.get_status()}")

In [15]:
f"batch_run_{run_version}"

## 9. View Results

Results are available in the **Snowsight AI Observability UI**:

1. Navigate to **AI & ML > Evaluations** in Snowsight
2. Filter by app name: `customer_support_langgraph_agent`
3. Select the run to inspect individual traces and metrics

You can also query the results programmatically:

In [24]:
print(f"Run Name:   {run_config.run_name}")
print(f"App Name:   {APP_NAME}")
print(f"App Version: {APP_VERSION}")
print(f"Final Status: {run.get_status()}")
print()
print("View full results in Snowsight:")
print("  → AI & ML > Evaluations")
print(f"  → Look for app '{APP_NAME}' version '{APP_VERSION}'")

In [29]:
eval_df = session.sql(f"""
    SELECT *
    FROM TABLE(
        SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
            'CUST_SUPPORT_DEMO', 'SUPPORT', '{APP_NAME}', 'EXTERNAL AGENT', '{run_config.run_name}'
        )
    )
    ORDER BY TIMESTAMP DESC
    LIMIT 50
""").to_pandas()

print(f"Evaluation records: {len(eval_df)}")
if not eval_df.empty:
    display(eval_df)
else:
    print("No evaluation data yet. Results may take a moment to appear.")
    print("Check Snowsight → AI & ML → Evaluations for the latest results.")

In [32]:
obs_event_sql = f"""SELECT * FROM TABLE(
    SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CUST_SUPPORT_DEMO', 
        'SUPPORT', 
        '{APP_NAME}', 
        'EXTERNAL AGENT')
) 
WHERE RECORD_ATTRIBUTES:"snow.ai.observability.run.name" = '{run_config.run_name}';"""

df = session.sql(obs_event_sql).to_pandas()

df

In [ ]:
df

## Next Steps

- **Iterate on the agent**: Try different LLMs (`llama3.1-70b`, `mistral-large`), adjust the system prompt, or add more tools
- **Compare versions**: Create new app versions and compare evaluation runs side-by-side in Snowsight
- **Add ground truth**: Include expected answers in the dataset to compute the `correctness` metric
- **Add more metrics**: Compute `coherence` or custom metrics for domain-specific evaluation
- **Production deployment**: Use the best-performing configuration to deploy as a Cortex Agent or Streamlit app